# 05 — Noise-Aware QAOA Depth Selection

**QuantumOptLab · Milestone 5 · Seed 31 · Self-contained research notebook**

**Research question:** Given a graph and a synthetic gate-noise condition, can a model choose between QAOA depths $p=1$ and $p=2$ so that the extra two-qubit gates are justified by improved Max-Cut solution quality?

This notebook extends Phase 4 without changing its stored results. It reads the **Phase 4 full-run graph catalog, ideal-optimized parameters, and frozen validation noise outcomes**. It generates the **missing noise outcomes on training graphs only**, trains policies using the training graphs, chooses the learned method using **grouped cross-validation on training graphs**, and performs an exploratory validation comparison.

A predeclared decision utility trades solution quality against two-qubit gate cost:

$$U(p;G,\eta)=\text{measured approximation ratio}(p;G,\eta)-\lambda\,N_{\mathrm{CX}}(p,G),\qquad \lambda=0.0005.$$

The penalty is a **declared engineering preference**, not a physical quantum error model or a claim of quantum advantage. For a graph with $m$ edges, our zero-optimization compilation convention yields $2mp$ CX gates. A depth-2 decision must therefore deliver a predicted ratio gain greater than $\lambda(2m)$.

**Data protocol:** 40 train / 20 development-validation / 20 locked-test graphs in full mode. The validation set was already inspected in Phase 4; it is **not an untouched final test**. The 20 locked-test graphs remain unused until Milestone 6. No results below should be called a blind generalization result.

**Prerequisites:** Phase 4 must already have finished in **full** mode in this project folder. Install `scikit-learn` in the existing Python (QuantumOptLab) environment if necessary. Set `RUN_MODE = "smoke"` for a small functional check before the full run. Each code cell is introduced by a Markdown explanation.

### 1. Configuration and Phase 4 artifact discovery

Use the existing QuantumOptLab root and its `reports` folders. We never rewrite Phase 4 files. Smoke mode keeps separate output names and checks only a small sample of the **training and validation** graphs while leaving the test set locked. The full mode trains on all 40 training graphs and reports performance on all 20 development-validation graphs.

In [ ]:
from __future__ import annotations

from pathlib import Path
from hashlib import sha256
from collections import Counter
import itertools
import json
import math
import platform
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error
import qiskit, qiskit_aer, sklearn

SEED = 31
RUN_MODE = "full"      # Switch to "smoke" for a small functionality test
assert RUN_MODE in ("full", "smoke")
SHOTS = 1024
REPEATS = 3
BATCH_SIZE = 8
LAMBDA_CX = 0.0005  # Approximation-ratio points deducted per compiled CX
NOISE_CONDITIONS = (
    ("ideal", 0.0, 0.0),
    ("cx_002", 0.02, 0.0),
    ("cx_005", 0.05, 0.0),
    ("cx_010", 0.10, 0.0),
    ("readout_002", 0.0, 0.02),
    ("cx_005_readout_002", 0.05, 0.02),
)
CONDITION_ORDER = [r[0] for r in NOISE_CONDITIONS]

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "notebooks").is_dir(), f"Open the notebook inside QuantumOptLab: {ROOT}"
EXPERIMENTS = ROOT / "reports" / "experiments"
FIGURES = ROOT / "reports" / "figures"
EXPERIMENTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
PREFIX = f"phase05_{RUN_MODE}_seed{SEED}"
PHASE4_PREFIX = f"phase04_full_seed{SEED}"
P4_PATHS = {
    "catalog": EXPERIMENTS / f"{PHASE4_PREFIX}_graph_catalog.json",
    "ideal": EXPERIMENTS / f"{PHASE4_PREFIX}_qaoa_ideal.csv",
    "validation_noise": EXPERIMENTS / f"{PHASE4_PREFIX}_noise_raw.csv",
    "manifest": EXPERIMENTS / f"{PHASE4_PREFIX}_manifest.json",
}
missing = [str(path) for path in P4_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing Phase 4 FULL-RUN artifacts; run notebook 04 in full mode first. Missing:\n"
        + "\n".join(missing)
    )

SOURCE_HASHES = {key: sha256(path.read_bytes()).hexdigest() for key, path in P4_PATHS.items()}
SETTINGS = {
    "seed": SEED, "mode": RUN_MODE, "shots": SHOTS, "repeats": REPEATS,
    "batch_size": BATCH_SIZE, "lambda_cx": LAMBDA_CX,
    "noise_conditions": NOISE_CONDITIONS, "phase4_source_sha256": SOURCE_HASHES,
    "basis_gates": ["h", "rx", "rz", "cx"], "optimization_level": 0,
    "policy_version": "phase05-groupcv-regret-v1",
}
CONFIG_HASH = sha256(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:12]
RUN_START = time.perf_counter()
print("Project:", ROOT)
print("Qiskit:", qiskit.__version__, "Aer:", qiskit_aer.__version__, "scikit-learn:", sklearn.__version__)
print("Phase 5:", RUN_MODE, "| configuration:", CONFIG_HASH)
print("Source: Phase 4 FULL run | test graph labels are not requested")


### 2. Validate the Phase 4 split and source provenance

Read the stored catalog rather than regenerating it with a potentially different NetworkX version. We restrict all subsequent graph reconstruction to **training and validation IDs**. Verify that the saved Phase 4 validation measurements include every specified condition and repeat, and that neither the ideal nor noisy outcome files contains locked-test graphs.

In [ ]:
with P4_PATHS["catalog"].open(encoding="utf-8") as f:
    catalog_doc = json.load(f)
with P4_PATHS["manifest"].open(encoding="utf-8") as f:
    p4_manifest = json.load(f)
catalog = pd.DataFrame(catalog_doc["records"])
ideal_source = pd.read_csv(P4_PATHS["ideal"])
validation_source = pd.read_csv(P4_PATHS["validation_noise"])

assert int(catalog_doc["seed"]) == SEED
assert p4_manifest.get("config_hash") == catalog_doc["config_hash"]
assert p4_manifest.get("phase") == 4
assert p4_manifest.get("config", {}).get("run_mode") == "full"
assert len(catalog) == 80 and catalog.graph_id.is_unique
assert catalog.split.value_counts().to_dict() == {"train": 40, "validation": 20, "test": 20}
train_all = sorted(catalog.loc[catalog.split == "train", "graph_id"])
validation_all = sorted(catalog.loc[catalog.split == "validation", "graph_id"])
LOCKED_TEST_IDS = set(catalog.loc[catalog.split == "test", "graph_id"])
assert not (set(ideal_source.graph_id) & LOCKED_TEST_IDS)
assert not (set(validation_source.graph_id) & LOCKED_TEST_IDS)
assert len(ideal_source) == 120
assert set(ideal_source.graph_id) == set(train_all + validation_all)
assert ideal_source.groupby("graph_id").depth_p.nunique().eq(2).all()
assert ideal_source.config_hash.eq(catalog_doc["config_hash"]).all()
assert validation_source.config_hash.eq(catalog_doc["config_hash"]).all()
assert len(validation_source) == 20 * 2 * len(NOISE_CONDITIONS) * REPEATS
assert set(validation_source.graph_id) == set(validation_all)
assert not validation_source.duplicated(["graph_id", "depth_p", "condition", "repeat"]).any()
assert validation_source.shots.eq(SHOTS).all()
assert set(validation_source.condition) == set(CONDITION_ORDER)
for cname, cx, readout in NOISE_CONDITIONS:
    subset = validation_source.loc[validation_source.condition == cname]
    assert np.allclose(subset.cx_p, cx) and np.allclose(subset.readout_p, readout)

if RUN_MODE == "smoke":
    train_ids = sorted(catalog.loc[catalog.split == "train"].groupby("family", sort=True).head(2).graph_id)
    validation_ids = sorted(catalog.loc[catalog.split == "validation"].groupby("family", sort=True).head(1).graph_id)
else:
    train_ids, validation_ids = train_all, validation_all
ACTIVE_IDS = set(train_ids + validation_ids)
assert set(train_ids).isdisjoint(validation_ids)
assert ACTIVE_IDS.isdisjoint(LOCKED_TEST_IDS)
assert len(train_ids) == (40 if RUN_MODE == "full" else 8)
assert len(validation_ids) == (20 if RUN_MODE == "full" else 4)

ideal_active = ideal_source.loc[ideal_source.graph_id.isin(ACTIVE_IDS)].copy()
validation_active = validation_source.loc[validation_source.graph_id.isin(validation_ids)].copy()
print("Training graphs:", len(train_ids), "| Development-validation graphs:", len(validation_ids))
print("Locked test graphs:", len(LOCKED_TEST_IDS), "| Still untouched")
print("Prior Phase 4 validation noisy records reused:", len(validation_active))
display(catalog.loc[catalog.graph_id.isin(ACTIVE_IDS)].groupby(["split", "family"]).size().unstack(fill_value=0))


### 3. Reconstruct only active graphs and precompute their exact cut values

Load the stored edge lists for the selected train and validation graphs. Exact Max-Cut values are used **only to measure** the reward of existing circuits; neither the model inputs nor the depth-selection policy will see the optimal cut or the optimized quantum objective. Check that reconstructed graph costs match Phase 4's saved optima.

In [ ]:
CATALOG_BY_ID = {row["graph_id"]: row for row in catalog_doc["records"]}
GRAPHS, COSTS, OPTIMA = {}, {}, {}
for graph_id in sorted(ACTIVE_IDS):
    row = CATALOG_BY_ID[graph_id]
    g = nx.Graph()
    g.add_nodes_from(range(int(row["n"])))
    g.add_weighted_edges_from((int(u), int(v), int(w)) for u, v, w in row["edge_list"])
    assert nx.is_connected(g) and g.number_of_edges() == int(row["edges"])
    assert nx.weisfeiler_lehman_graph_hash(g, edge_attr="weight") == row["fingerprint"]
    positions = np.arange(1 << int(row["n"]), dtype=np.uint64)
    costs = np.zeros(len(positions), dtype=np.float64)
    for u, v, data in g.edges(data=True):
        costs += float(data["weight"]) * (((positions >> u) ^ (positions >> v)) & 1)
    optimum = float(costs.max())
    source_optima = ideal_active.loc[ideal_active.graph_id == graph_id, "optimum"].to_numpy()
    assert len(source_optima) == 2 and np.allclose(source_optima, optimum)
    GRAPHS[graph_id], COSTS[graph_id], OPTIMA[graph_id] = g, costs, optimum
assert len(GRAPHS) == len(ACTIVE_IDS)
assert LOCKED_TEST_IDS.isdisjoint(GRAPHS)
print("Verified graph structures and exact cut costs for", len(GRAPHS), "active graphs")


### 4. Compile the frozen QAOA parameters for *training* circuits

Preserve Phase 4's $RZZ(-\gamma w)$ cost gates, $RX(2\beta)$ mixers, CX-based compilation, and optimization level zero. The parameters remain fixed; Phase 5 does not reoptimize a circuit for its noise environment. Count CX gates to enforce a reproducible circuit-cost proxy.

In [ ]:
def qaoa_circuit(g, params, depth):
    theta = np.asarray(params, dtype=float)
    assert theta.shape == (2 * depth,)
    qc = QuantumCircuit(g.number_of_nodes())
    qc.h(range(g.number_of_nodes()))
    for gamma, beta in zip(theta[:depth], theta[depth:]):
        for u, v, data in g.edges(data=True):
            qc.rzz(-float(gamma) * float(data["weight"]), u, v)
        for qubit in range(g.number_of_nodes()):
            qc.rx(2 * float(beta), qubit)
    return qc

COMPILED = {}
RESOURCE_ROWS = []
for graph_id in train_ids:
    graph = GRAPHS[graph_id]
    for depth in (1, 2):
        record = ideal_active.loc[(ideal_active.graph_id == graph_id) & (ideal_active.depth_p == depth)].iloc[0]
        params = json.loads(record.params_json)
        qc = qaoa_circuit(graph, params, depth)
        compiled = transpile(qc, basis_gates=["h", "rx", "rz", "cx"],
                             optimization_level=0, seed_transpiler=SEED)
        count_cx = int(compiled.count_ops().get("cx", 0))
        assert count_cx == 2 * graph.number_of_edges() * depth
        assert "rzz" not in compiled.count_ops()
        compiled.measure_all()
        COMPILED[(graph_id, depth)] = compiled
        RESOURCE_ROWS.append({"graph_id": graph_id, "depth_p": depth,
                              "cx_gates": count_cx, "compiled_depth": int(compiled.depth())})
resources = pd.DataFrame(RESOURCE_ROWS)
resources.to_csv(EXPERIMENTS / f"{PREFIX}_train_circuit_resources.csv", index=False)
print("Compiled", len(COMPILED), "train circuits from saved Phase 4 parameters")
display(resources.groupby("depth_p")[["cx_gates", "compiled_depth"]].mean().round(2))


### 5. Generate and checkpoint the *missing training* noise outcomes

Apply Phase 4's exact synthetic CX depolarizing and symmetric readout-error models at the same six conditions, three repetitions, and 1,024 shots. Results are written incrementally to a new Phase 5 CSV. Seed each fixed **batch** deterministically so an interrupted run can safely resume without changing the other saved measurements. The cached validation outcomes are reused as-is; no test circuit is executed.

In [ ]:
def make_noise_model(cx_p, readout_p):
    model = NoiseModel()
    if cx_p:
        model.add_all_qubit_quantum_error(depolarizing_error(float(cx_p), 2), ["cx"])
    if readout_p:
        rate = float(readout_p)
        model.add_all_qubit_readout_error(ReadoutError([[1-rate, rate], [rate, 1-rate]]))
    return model


def sample_cut_stats(graph_id, counts):
    costs = COSTS[graph_id]
    optimal = OPTIMA[graph_id]
    shots = sum(int(count) for count in counts.values())
    weighted_sum = 0.0
    optimal_samples = 0
    for bitstring, repetitions in counts.items():
        x = int(bitstring.replace(" ", ""), 2)
        weighted_sum += float(costs[x]) * int(repetitions)
        optimal_samples += int(repetitions) * bool(np.isclose(costs[x], optimal))
    return weighted_sum / shots, optimal_samples / shots, shots

TRAIN_RAW_PATH = EXPERIMENTS / f"{PREFIX}_train_noise_raw.csv"
if TRAIN_RAW_PATH.exists():
    train_rows = pd.read_csv(TRAIN_RAW_PATH).to_dict("records")
    assert all(row["config_hash"] == CONFIG_HASH for row in train_rows), (
        "Phase 5 checkpoint configuration mismatch. Keep the old settings or rename the existing Phase 5 checkpoint."
    )
else:
    train_rows = []
completed = {(row["graph_id"], int(row["depth_p"]), row["condition"], int(row["repeat"]))
             for row in train_rows}
assert not len(completed) < len(train_rows), "Duplicate entries in the training checkpoint"
assert set(row["graph_id"] for row in train_rows).issubset(train_ids)

ordered_pairs = list(itertools.product(train_ids, (1, 2)))
noise_started = time.perf_counter()
for condition_idx, (condition, cx_p, readout_p) in enumerate(NOISE_CONDITIONS):
    simulator = AerSimulator(method="density_matrix", noise_model=make_noise_model(cx_p, readout_p),
                             max_parallel_experiments=1)
    for repeat in range(REPEATS):
        for offset in range(0, len(ordered_pairs), BATCH_SIZE):
            batch = ordered_pairs[offset:offset + BATCH_SIZE]
            pending = [pair for pair in batch if (pair[0], pair[1], condition, repeat) not in completed]
            if not pending:
                continue
            simulator_seed = SEED + 2_000_000 + condition_idx * 100_000 + repeat * 1000 + offset // BATCH_SIZE
            # Evaluate the ORIGINAL fixed batch on restart; this preserves per-circuit RNG ordering.
            result = simulator.run([COMPILED[pair] for pair in batch],
                                   shots=SHOTS, seed_simulator=simulator_seed).result()
            assert result.success, f"Aer failed for {condition} repeat {repeat}"
            for index, (graph_id, depth) in enumerate(batch):
                key = (graph_id, depth, condition, repeat)
                if key in completed:
                    continue
                expected, prob_opt, shots = sample_cut_stats(graph_id, result.get_counts(index))
                assert shots == SHOTS
                train_rows.append({
                    "config_hash": CONFIG_HASH, "graph_id": graph_id,
                    "depth_p": depth, "condition": condition,
                    "cx_p": cx_p, "readout_p": readout_p,
                    "repeat": repeat, "simulator_seed": simulator_seed,
                    "shots": shots, "expected_cut_shots": expected,
                    "approx_ratio": expected / OPTIMA[graph_id], "p_optimal": prob_opt,
                })
                completed.add(key)
            pd.DataFrame(train_rows).to_csv(TRAIN_RAW_PATH, index=False)
    print(f"Training condition {condition}: {len(train_ids) * 2 * REPEATS} paired-depth records ready")

train_raw = pd.DataFrame(train_rows)
expected_train = len(train_ids) * 2 * len(NOISE_CONDITIONS) * REPEATS
assert len(train_raw) == expected_train
assert not train_raw.duplicated(["graph_id", "depth_p", "condition", "repeat"]).any()
assert train_raw.shots.eq(SHOTS).all()
assert LOCKED_TEST_IDS.isdisjoint(set(train_raw.graph_id))
print("Training noise samples:", len(train_raw), "| Noise elapsed:",
      round(time.perf_counter() - noise_started, 1), "seconds")


### 6. Build paired graph-condition labels and allowable input features

Average the three simulator repeats **within each graph / condition / depth**, then pair $p=1$ with $p=2$. The target is their measured approximation-ratio difference $\Delta R = R_2-R_1$. The optimal utility decision compares $\Delta R$ to $\lambda(N_{CX,2}-N_{CX,1})$.

The model sees only **pre-execution graph topology, edge weights, noise settings, and theoretical CX resources**. It does **not** receive optimized circuit angles, expected QAOA quality, Max-Cut optimum, graph ID, or outcome measurements as input features. Grouping observations by graph prevents condition-level leakage during cross-validation.

In [ ]:
def graph_features(graph_id):
    g = GRAPHS[graph_id]
    degrees = np.array([degree for _, degree in g.degree()], dtype=float)
    weights = np.array([float(d["weight"]) for *_, d in g.edges(data=True)], dtype=float)
    n, edges = g.number_of_nodes(), g.number_of_edges()
    return {
        "graph_id": graph_id, "n": n, "edges": edges,
        "density": nx.density(g), "average_degree": float(degrees.mean()),
        "degree_std": float(degrees.std()),
        "clustering": float(nx.average_clustering(g)),
        "triangles": float(sum(nx.triangles(g).values()) / 3),
        "weight_sum": float(weights.sum()), "weight_mean": float(weights.mean()),
        "weight_std": float(weights.std()),
        "max_weight": float(weights.max()),
        "weighted_flag": int(not np.allclose(weights, 1.0)),
    }

metadata = pd.DataFrame([graph_features(i) for i in sorted(ACTIVE_IDS)])
train_raw = train_raw.assign(split="train")
validation_raw = validation_active.copy().assign(split="validation")
all_raw = pd.concat([train_raw, validation_raw], ignore_index=True)
assert all_raw.groupby(["graph_id", "condition", "depth_p"]).size().eq(REPEATS).all()

summarized = all_raw.groupby(
    ["graph_id", "split", "condition", "cx_p", "readout_p", "depth_p"], as_index=False
).agg(ratio=("approx_ratio", "mean"), p_optimal=("p_optimal", "mean"))
paired = summarized.pivot(index=["graph_id", "split", "condition", "cx_p", "readout_p"],
                           columns="depth_p", values="ratio").reset_index()
paired.columns = [f"ratio_p{int(k)}" if isinstance(k, (int, np.integer)) else k for k in paired.columns]
assert {"ratio_p1", "ratio_p2"}.issubset(paired.columns)
paired = paired.merge(metadata, on="graph_id", validate="many_to_one")
paired["cx_p1"] = 2 * paired.edges
paired["cx_p2"] = 4 * paired.edges
paired["delta_cx"] = paired.cx_p2 - paired.cx_p1
paired["cx_exposure_p1"] = paired.cx_p * paired.cx_p1
paired["cx_exposure_p2"] = paired.cx_p * paired.cx_p2
paired["readout_exposure"] = paired.readout_p * paired.n
paired["delta_ratio"] = paired.ratio_p2 - paired.ratio_p1
paired["utility_p1"] = paired.ratio_p1 - LAMBDA_CX * paired.cx_p1
paired["utility_p2"] = paired.ratio_p2 - LAMBDA_CX * paired.cx_p2
paired["delta_utility"] = paired.utility_p2 - paired.utility_p1
paired["oracle_depth"] = np.where(paired.delta_utility > 1e-12, 2, 1)

FEATURES = [
    "n", "edges", "density", "average_degree", "degree_std", "clustering",
    "triangles", "weight_sum", "weight_mean", "weight_std", "max_weight",
    "weighted_flag", "cx_p", "readout_p", "cx_p1", "cx_p2",
    "cx_exposure_p1", "cx_exposure_p2", "readout_exposure",
]
assert not set(FEATURES) & {"graph_id", "split", "ratio_p1", "ratio_p2", "delta_ratio",
                             "delta_utility", "oracle_depth", "utility_p1", "utility_p2"}
train_pairs = paired.loc[paired.split == "train"].sort_values(["graph_id", "condition"]).reset_index(drop=True)
val_pairs = paired.loc[paired.split == "validation"].sort_values(["graph_id", "condition"]).reset_index(drop=True)
assert len(train_pairs) == len(train_ids) * len(NOISE_CONDITIONS)
assert len(val_pairs) == len(validation_ids) * len(NOISE_CONDITIONS)
assert train_pairs[FEATURES].notna().all().all() and val_pairs[FEATURES].notna().all().all()
assert train_pairs[FEATURES].apply(np.isfinite).all().all()
assert val_pairs[FEATURES].apply(np.isfinite).all().all()
paired.to_csv(EXPERIMENTS / f"{PREFIX}_paired_labels.csv", index=False)
print("Train decisions:", len(train_pairs), "| Development-validation decisions:", len(val_pairs))
display(train_pairs.groupby("condition").agg(mean_depth_benefit=("delta_ratio", "mean"),
                                            oracle_p2_fraction=("oracle_depth", lambda x: (x == 2).mean())).round(4))


### 7. Define baselines and select an adaptive model using *training groups only*

Compare fixed depths, a **training-only noise-condition lookup**, and a simple gate-exposure threshold with Ridge and Random Forest regression. The regression target is the ratio difference $\Delta R$. For Ridge and Random Forest, choose hyperparameters using **four-fold grouped cross-validation on training graphs** and minimize decision regret in the declared utility. Choose the final learned model **only by its training cross-validation result**. The validation graph outcomes are not used to fit or select this model.

In [ ]:
def decisions_from_delta(data, predicted_delta):
    pred = np.asarray(predicted_delta, dtype=float)
    threshold = LAMBDA_CX * data["delta_cx"].to_numpy(dtype=float)
    return np.where(pred > threshold + 1e-12, 2, 1)


def row_regret(data, depths):
    depths = np.asarray(depths, dtype=int)
    chosen = np.where(depths == 2, data.utility_p2.to_numpy(), data.utility_p1.to_numpy())
    return np.maximum(data.utility_p1.to_numpy(), data.utility_p2.to_numpy()) - chosen

X_train = train_pairs[FEATURES]
y_train = train_pairs.delta_ratio.to_numpy(dtype=float)
groups = train_pairs.graph_id.to_numpy()
folds = list(GroupKFold(n_splits=4).split(X_train, y_train, groups))
assert all(set(train_pairs.iloc[train_idx].graph_id).isdisjoint(
    set(train_pairs.iloc[val_idx].graph_id)) for train_idx, val_idx in folds)

CANDIDATES = {}
for alpha in (0.1, 1.0, 10.0, 100.0):
    CANDIDATES[f"ridge_alpha_{alpha:g}"] = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
for leaf in (2, 5, 10):
    CANDIDATES[f"forest_leaf_{leaf}"] = RandomForestRegressor(
        n_estimators=160, min_samples_leaf=leaf, random_state=SEED, n_jobs=-1
    )

cv_rows = []
for name, prototype in CANDIDATES.items():
    out_of_fold = np.full(len(train_pairs), np.nan)
    for train_idx, holdout_idx in folds:
        fitted = clone(prototype).fit(X_train.iloc[train_idx], y_train[train_idx])
        out_of_fold[holdout_idx] = fitted.predict(X_train.iloc[holdout_idx])
    assert np.isfinite(out_of_fold).all()
    predicted_depth = decisions_from_delta(train_pairs, out_of_fold)
    cv_rows.append({
        "candidate": name,
        "family": "ridge" if name.startswith("ridge") else "random_forest",
        "cv_mean_regret": float(row_regret(train_pairs, predicted_depth).mean()),
        "cv_delta_mae": float(np.abs(out_of_fold - y_train).mean()),
        "cv_choose_p2_fraction": float(np.mean(predicted_depth == 2)),
    })
cv_table = pd.DataFrame(cv_rows).sort_values(
    ["cv_mean_regret", "cv_delta_mae", "candidate"], kind="stable"
).reset_index(drop=True)
cv_table.to_csv(EXPERIMENTS / f"{PREFIX}_train_groupcv.csv", index=False)

ridge_winner = cv_table.loc[cv_table.family == "ridge"].iloc[0]["candidate"]
forest_winner = cv_table.loc[cv_table.family == "random_forest"].iloc[0]["candidate"]
selected_name = str(cv_table.iloc[0]["candidate"])
FITTED = {name: clone(CANDIDATES[name]).fit(X_train, y_train)
          for name in [ridge_winner, forest_winner]}
assert selected_name in FITTED

noise_lookup = train_pairs.groupby("condition").delta_ratio.mean().to_dict()
gate_thresholds = [0.0, 0.05, 0.1, 0.2, 0.4, 0.8, 1.2, 2.0, float("inf")]
gate_search = []
for threshold in gate_thresholds:
    proposed = np.where(train_pairs.cx_exposure_p1.to_numpy() < threshold, 2, 1)
    gate_search.append({"threshold": threshold, "mean_train_regret": row_regret(train_pairs, proposed).mean(),
                        "mean_train_cx": float(np.where(proposed == 2, train_pairs.cx_p2, train_pairs.cx_p1).mean())})
gate_search = pd.DataFrame(gate_search).sort_values(
    ["mean_train_regret", "mean_train_cx", "threshold"], kind="stable"
)
gate_threshold = float(gate_search.iloc[0].threshold)

print("Selected via training GroupKFold:", selected_name)
print("Best Ridge:", ridge_winner, "| Best Random Forest:", forest_winner)
print("Train-tuned gate exposure threshold:", gate_threshold)
display(cv_table.round(5))


### 8. Freeze and compare all selection policies on development-validation graphs

Apply the chosen training-fitted models **once** to the development-validation feature table. The rules can use the graph and known noise parameters, but cannot look at the outcomes of those validation graphs. The oracle policy *does* use the observed outcomes; it is included only as an unattainable upper bound.

The primary endpoint is **mean decision utility** (higher is better), accompanied by regret, approximation ratio, CX count, and how often depth 2 was selected. Fixed-depth baselines remain visible even if the adaptive method does not improve on them.

In [ ]:
PREDICTIONS = {
    "always_p1": np.ones(len(val_pairs), dtype=int),
    "always_p2": np.full(len(val_pairs), 2, dtype=int),
    "train_noise_lookup": decisions_from_delta(
        val_pairs, val_pairs.condition.map(noise_lookup).to_numpy(dtype=float)
    ),
    "gate_budget_rule": np.where(val_pairs.cx_exposure_p1.to_numpy() < gate_threshold, 2, 1),
    "ridge": decisions_from_delta(val_pairs, FITTED[ridge_winner].predict(val_pairs[FEATURES])),
    "random_forest": decisions_from_delta(val_pairs, FITTED[forest_winner].predict(val_pairs[FEATURES])),
    "selected_cv_model": decisions_from_delta(val_pairs, FITTED[selected_name].predict(val_pairs[FEATURES])),
    "oracle_upper_bound": val_pairs.oracle_depth.to_numpy(dtype=int),
}

PREDICTION_DELTAS = {
    "train_noise_lookup": val_pairs.condition.map(noise_lookup).to_numpy(dtype=float),
    "ridge": FITTED[ridge_winner].predict(val_pairs[FEATURES]),
    "random_forest": FITTED[forest_winner].predict(val_pairs[FEATURES]),
    "selected_cv_model": FITTED[selected_name].predict(val_pairs[FEATURES]),
}

policy_records = []
for policy, choices in PREDICTIONS.items():
    assert set(np.unique(choices)).issubset({1, 2})
    for index, record in val_pairs.iterrows():
        chosen = int(choices[index])
        base = record.ratio_p2 if chosen == 2 else record.ratio_p1
        cx = int(record.cx_p2 if chosen == 2 else record.cx_p1)
        reward = float(base - LAMBDA_CX * cx)
        assert abs(reward - (record.utility_p2 if chosen == 2 else record.utility_p1)) < 1e-12
        policy_records.append({
            "policy": policy, "graph_id": record.graph_id, "family": CATALOG_BY_ID[record.graph_id]["family"],
            "n": int(record.n), "condition": record.condition,
            "cx_p": float(record.cx_p), "readout_p": float(record.readout_p),
            "selected_depth": chosen, "oracle_depth": int(record.oracle_depth),
            "mean_ratio": float(base), "utility": reward, "cx_gates": cx,
            "regret": max(record.utility_p1, record.utility_p2) - reward,
            "ratio_p1": float(record.ratio_p1), "ratio_p2": float(record.ratio_p2),
        })
policy_df = pd.DataFrame(policy_records)
assert len(policy_df) == len(val_pairs) * len(PREDICTIONS)
assert policy_df.regret.ge(-1e-10).all()
policy_df.to_csv(EXPERIMENTS / f"{PREFIX}_development_policy_decisions.csv", index=False)

summary = policy_df.groupby("policy", as_index=False).agg(
    n_decisions=("utility", "count"), mean_utility=("utility", "mean"),
    mean_ratio=("mean_ratio", "mean"), mean_regret=("regret", "mean"),
    mean_cx_gates=("cx_gates", "mean"),
    fraction_p2=("selected_depth", lambda x: float((x == 2).mean())),
    oracle_depth_accuracy=("selected_depth", lambda x: float((x == policy_df.loc[x.index, "oracle_depth"]).mean())),
).sort_values(["mean_utility", "mean_cx_gates"], ascending=[False, True])
summary.to_csv(EXPERIMENTS / f"{PREFIX}_development_policy_summary.csv", index=False)
print("Frozen learned model:", selected_name)
print("All results below are development-validation, not locked-test results.")
display(summary.round(4))


### 9. Estimate uncertainty by resampling **graphs**, not graph-condition rows

Each validation graph appears in six noise conditions, so those rows are not six independent graph draws. Average within graph first, then bootstrap the **graph-level paired utility difference** between the cross-validated selected model and each deployable baseline. These intervals quantify variation across the 20 development-validation graph instances; they do not account for circuit-parameter optimization uncertainty or make this a held-out test.

In [ ]:
selected_val = policy_df.loc[policy_df.policy == "selected_cv_model", ["graph_id", "condition", "utility"]].rename(
    columns={"utility": "selected_utility"}
)
ci_rows = []
for j, policy in enumerate(["always_p1", "always_p2", "train_noise_lookup", "gate_budget_rule", "ridge", "random_forest"]):
    comparator = policy_df.loc[policy_df.policy == policy, ["graph_id", "condition", "utility"]]
    compare = selected_val.merge(comparator, on=["graph_id", "condition"], validate="one_to_one")
    differences = compare.assign(delta=lambda x: x.selected_utility - x.utility).groupby("graph_id").delta.mean()
    assert len(differences) == len(validation_ids)
    rng = np.random.default_rng(SEED + 100 + j)
    samples = rng.choice(differences.to_numpy(), size=(4000, len(differences)), replace=True).mean(axis=1)
    ci_rows.append({
        "selected_minus": policy, "n_graphs": len(differences),
        "mean_utility_difference": float(differences.mean()),
        "ci95_low": float(np.quantile(samples, 0.025)),
        "ci95_high": float(np.quantile(samples, 0.975)),
    })
ci_table = pd.DataFrame(ci_rows)
ci_table.to_csv(EXPERIMENTS / f"{PREFIX}_graph_bootstrap_comparisons.csv", index=False)
display(ci_table.round(5))

by_condition = policy_df.groupby(["policy", "condition"], as_index=False).agg(
    mean_utility=("utility", "mean"), mean_ratio=("mean_ratio", "mean"),
    mean_regret=("regret", "mean"), fraction_p2=("selected_depth", lambda x: (x == 2).mean()),
    mean_cx_gates=("cx_gates", "mean"),
)
by_condition.to_csv(EXPERIMENTS / f"{PREFIX}_policy_by_condition.csv", index=False)
print("Utility baseline comparison is paired at graph level; no test outcomes were evaluated.")


### 10. Visualize the selector's achieved quality and quantum circuit cost

Show average development-validation utility, expected approximation ratio, regret, and CX gate usage for deployable methods. The oracle upper bound is displayed only as an analysis reference. In the second figure, compare the fraction of graphs assigned to depth 2 by the selected model and by the true utility-maximizing oracle under each noise condition.

In [ ]:
policies_plot = ["always_p1", "always_p2", "train_noise_lookup", "gate_budget_rule", "ridge", "random_forest", "selected_cv_model", "oracle_upper_bound"]
plot_summary = summary.set_index("policy").loc[policies_plot]
labels_plot = ["Fixed p=1", "Fixed p=2", "Noise lookup", "Gate rule", "Ridge", "Forest", "CV selected", "Oracle"]
bar_colors = ["#8A9BA8", "#8A9BA8", "#9AADB5", "#9AADB5", "#5C8CA8", "#5C8CA8", "#2C7791", "#AAA58F"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
x = np.arange(len(policies_plot))
axes[0].bar(x, plot_summary.mean_utility, color=bar_colors)
axes[0].set_ylabel("Mean decision utility (higher is better)")
axes[0].set_title("Held-out-by-Graph Development Validation")
axes[0].set_xticks(x, labels_plot, rotation=37, ha="right")
axes[0].grid(axis="y", alpha=0.18)
axes[1].scatter(plot_summary.mean_cx_gates, plot_summary.mean_ratio,
                s=[75 if p == "selected_cv_model" else 43 for p in policies_plot],
                color=bar_colors)
for name, px, py in zip(labels_plot, plot_summary.mean_cx_gates, plot_summary.mean_ratio):
    axes[1].annotate(name, (px, py), xytext=(3, 4), textcoords="offset points", fontsize=7)
axes[1].set_xlabel("Mean compiled CX gates")
axes[1].set_ylabel("Mean approximation ratio")
axes[1].set_title("Solution Quality vs Circuit Cost")
axes[1].grid(alpha=0.18)
fig.tight_layout()
FIG_POLICY = FIGURES / f"{PREFIX}_policy_comparison.png"
fig.savefig(FIG_POLICY, dpi=170, bbox_inches="tight")
plt.show()

chosen_condition = by_condition[by_condition.policy == "selected_cv_model"].set_index("condition").reindex(CONDITION_ORDER)
oracle_condition = by_condition[by_condition.policy == "oracle_upper_bound"].set_index("condition").reindex(CONDITION_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(10.7, 3.8))
x = np.arange(len(CONDITION_ORDER))
axes[0].bar(x - .19, chosen_condition.fraction_p2, width=.36, label="CV-selected", color="#2C7791")
axes[0].bar(x + .19, oracle_condition.fraction_p2, width=.36, label="Oracle", color="#ABA585")
axes[0].set_xticks(x, CONDITION_ORDER, rotation=35, ha="right")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Fraction choosing p=2")
axes[0].set_title("Depth Decisions by Noise Condition")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", alpha=0.2)
axes[1].bar(x, chosen_condition.mean_regret, color="#5C8CA8")
axes[1].set_xticks(x, CONDITION_ORDER, rotation=35, ha="right")
axes[1].set_ylabel("Mean regret vs oracle (lower is better)")
axes[1].set_title("Selector Regret by Condition")
axes[1].grid(axis="y", alpha=0.2)
fig.tight_layout()
FIG_DECISIONS = FIGURES / f"{PREFIX}_decision_by_noise.png"
fig.savefig(FIG_DECISIONS, dpi=170, bbox_inches="tight")
plt.show()
print("Saved figures:", FIG_POLICY.relative_to(ROOT), "and", FIG_DECISIONS.relative_to(ROOT))


### 11. Freeze the trained selector and pre-register the Milestone 6 test protocol

Save the winning fitted model, its allowable feature order, its exact hyperparameters, the graph-level cross-validation outcomes, and the engineering cost penalty. The artifact is **development-only** until M6. The locked-test data must not be evaluated or used to revise this policy, including after seeing an unfavorable result. If M6 changes any policy or objective, it requires a clearly identified new experimental version and a new test protocol.

In [ ]:
import joblib
MODEL_PATH = EXPERIMENTS / f"{PREFIX}_frozen_selector.joblib"
joblib.dump(FITTED[selected_name], MODEL_PATH)

model_card = {
    "project": "QuantumOptLab", "milestone": 5, "status": "frozen for M6 evaluation",
    "run_mode": RUN_MODE, "seed": SEED, "config_hash": CONFIG_HASH,
    "phase4_source_sha256": SOURCE_HASHES,
    "selected_model_name": selected_name,
    "selected_model_params": FITTED[selected_name].get_params(deep=False),
    "ridge_candidate": ridge_winner, "forest_candidate": forest_winner,
    "features": FEATURES,
    "target": "mean_qaoa_approx_ratio_depth2_minus_depth1",
    "decision": "choose p=2 iff predicted_delta_ratio > lambda_cx * (cx_p2 - cx_p1); else p=1",
    "lambda_cx": LAMBDA_CX,
    "gate_threshold": gate_threshold,
    "train_noise_lookup": {k: float(v) for k, v in noise_lookup.items()},
    "train_graph_count": len(train_ids), "development_validation_graph_count": len(validation_ids),
    "locked_test_graph_count": len(LOCKED_TEST_IDS),
    "policy_selection": "4-fold GroupKFold on training graphs, minimum mean utility regret",
    "evaluation_status": "Exploratory validation only; 20 untouched test graphs require M6",
}
CARD_PATH = EXPERIMENTS / f"{PREFIX}_frozen_selector_card.json"
CARD_PATH.write_text(json.dumps(model_card, indent=2, default=str), encoding="utf-8")
print("Saved frozen selector:", MODEL_PATH)
print("Saved policy card:", CARD_PATH)


### 12. Final integrity checks and reproducibility manifest

Verify complete simulated training labels, exact split boundaries, predeclared features, grouped cross-validation, valid regret calculations, and saved figures. The concluding summary is deliberately conditional: **a development-validation improvement is not an unbiased test finding**, and negative results are equally valid evidence.

In [ ]:
chosen = summary.loc[summary.policy == "selected_cv_model"].iloc[0]
p1 = summary.loc[summary.policy == "always_p1"].iloc[0]
p2 = summary.loc[summary.policy == "always_p2"].iloc[0]
oracle = summary.loc[summary.policy == "oracle_upper_bound"].iloc[0]

checks = {
    "Original Phase 4 full manifest and catalog match": p4_manifest["config_hash"] == catalog_doc["config_hash"],
    "Training/validation/test graph boundaries preserved": (len(train_all), len(validation_all), len(LOCKED_TEST_IDS)) == (40, 20, 20),
    "Every active graph reconstructed without test data": set(GRAPHS) == ACTIVE_IDS and LOCKED_TEST_IDS.isdisjoint(GRAPHS),
    "All missing training noise runs generated once": len(train_raw) == expected_train and not train_raw.duplicated(["graph_id", "depth_p", "condition", "repeat"]).any(),
    "Original Phase 4 validation outcomes reused": len(validation_active) == len(validation_ids) * 2 * len(NOISE_CONDITIONS) * REPEATS,
    "No locked-test outcome labels anywhere": LOCKED_TEST_IDS.isdisjoint(set(all_raw.graph_id) | set(ideal_active.graph_id) | set(policy_df.graph_id)),
    "Only graph/noise/resource inputs used for model": not set(FEATURES) & {"graph_id", "split", "delta_ratio", "ratio_p1", "ratio_p2", "oracle_depth", "optimum", "params_json"},
    "Cross-validation folds separate whole graph groups": all(set(train_pairs.iloc[ti].graph_id).isdisjoint(set(train_pairs.iloc[vi].graph_id)) for ti, vi in folds),
    "All graph-condition outcomes have paired depths": len(paired) == (len(train_ids) + len(validation_ids)) * len(NOISE_CONDITIONS),
    "CX compilation preserves Phase 4 convention": all(int(r.cx_gates) == 2 * GRAPHS[r.graph_id].number_of_edges() * r.depth_p for r in resources.itertuples()),
    "Selection policy defined by train cross-validation": selected_name == cv_table.iloc[0]["candidate"],
    "All policies have finite measured rewards": bool(np.isfinite(policy_df[["utility", "mean_ratio", "regret", "cx_gates"]].to_numpy()).all()),
    "Oracle upper bound respected": bool(chosen.mean_utility <= oracle.mean_utility + 1e-12),
    "Primary cost/reward arithmetic consistent": bool(np.allclose(policy_df.utility, policy_df.mean_ratio - LAMBDA_CX * policy_df.cx_gates)),
    "Final artifacts created": all(path.exists() for path in [TRAIN_RAW_PATH, MODEL_PATH, CARD_PATH, FIG_POLICY, FIG_DECISIONS]),
}
for name, ok in checks.items():
    print(("PASS" if ok else "FAIL") + " | " + name)
assert all(checks.values()), "Phase 5 integrity checks failed; investigate before freezing these results"

manifest = {
    "project": "QuantumOptLab", "phase": 5, "run_mode": RUN_MODE,
    "config": SETTINGS, "config_hash": CONFIG_HASH,
    "source_files": {name: str(path.relative_to(ROOT)) for name, path in P4_PATHS.items()},
    "versions": {"python": platform.python_version(), "qiskit": qiskit.__version__,
                 "qiskit_aer": qiskit_aer.__version__, "sklearn": sklearn.__version__,
                 "numpy": np.__version__, "networkx": nx.__version__},
    "n_train_graphs": len(train_ids), "n_development_graphs": len(validation_ids),
    "n_test_graphs_reserved": len(LOCKED_TEST_IDS),
    "train_noise_samples": len(train_raw), "reused_validation_samples": len(validation_active),
    "selected_model": selected_name, "utility_lambda_cx": LAMBDA_CX,
    "selected_mean_utility": float(chosen.mean_utility),
    "fixed_p1_mean_utility": float(p1.mean_utility),
    "fixed_p2_mean_utility": float(p2.mean_utility),
    "oracle_mean_utility": float(oracle.mean_utility),
    "all_checks": checks,
    "caveats": [
        "Phase 4 development validation was previously inspected; this is not a blind test.",
        "20 reserved test graph instances have no generated circuit outcomes.",
        "QAOA parameters were optimized under ideal noise assumptions; no noise-aware re-optimization occurred.",
        "CX penalty is a user-chosen proxy, not an estimated monetary or hardware cost.",
        "Synthetic Qiskit Aer noise is not a calibrated real-device noise model.",
        "Training grouped cross-validation optimizes regret; confidence intervals resample validation graphs.",
        "No quantum advantage, independent hardware performance, or final generalization claim is established.",
    ],
    "elapsed_seconds": round(time.perf_counter() - RUN_START, 2),
}
MANIFEST_PATH = EXPERIMENTS / f"{PREFIX}_manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
print(f"\nPASS: {sum(checks.values())}/{len(checks)} Phase 5 integrity checks")
print("Model:", selected_name)
print("Selected utility:", round(float(chosen.mean_utility), 5),
      "| Always p=1:", round(float(p1.mean_utility), 5),
      "| Always p=2:", round(float(p2.mean_utility), 5))
print("Saved manifest:", MANIFEST_PATH)
print("LOCKED TEST: 20 graphs reserved; no test outcomes generated.")
